# Notebook 11: Gate S0c — Does the Model Use the Demonstration Labels?

**Purpose**: Analyse the demonstration-label-corruption gate on
`Llama-3.1-8B-Instruct`. Answers the question that has to precede any protocol
comparison, and turns up a second finding that outranks it.

## Why this gate runs before the protocol grid

Every demonstration-selection protocol works by choosing *which labelled rows* to
show. If the model does not use the input→label mapping at all, then no such
protocol can help, and the entire grid has a zero ceiling. That is the Min et al.
(2022) versus Yoo et al. (2022) question — asked of *these* models on *these*
tasks rather than inherited from the literature.

The test: flip a fraction of the demonstration labels. At 100% corruption every
label is wrong. If the model is using the mapping, performance must move.

## Collecting the data

```bash
cd sata-project
PYTHONPATH=. python scripts/run_real_arm_grid.py --mode corruption-gate \
    --cache-dir _screen_cache --model Llama-3.1-8B-Instruct \
    --out results/v2/gate_corruption --tensor-parallel 1 --max-model-len 4096 \
    --shard 0 --n-shards 2       # shard 1 in parallel on the second GPU
```

`random x balanced` at corruption 0/50/100% plus a zero-shot arm, 4 datasets,
5 seeds, 250 OOD queries per unit, contextual-calibration logprobs captured.
64 units, 16,000 rows. Merged into `results/v2/gate_corruption_merged.parquet`.

**Inference lives in `scripts/`, not in this notebook, deliberately** — the run
is long and sharded, and a dying notebook kernel loses everything since the last
manual save. v1's git history carries a dozen consecutive "NB06 sync: periodic
checkpoint of in-progress results" commits, which is a notebook fighting its own
lack of resumability.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

pd.set_option("display.width", 220)
RESULTS = PROJECT_ROOT / "results" / "v2"


def margin(df):
    """Label-logprob margin: the model's decision variable before thresholding."""
    return df.logprob_1 - df.logprob_0


def p_positive(df):
    """Implied P(positive) = sigmoid(margin)."""
    return 1.0 / (1.0 + np.exp(df.logprob_0 - df.logprob_1))


def integrity(df, name=""):
    """A failed label-token lookup defaults to -100 and a failed decode to -1.
    Both would masquerade as findings, so check before interpreting anything."""
    bad_lp = ((df.logprob_0 == -100) | (df.logprob_1 == -100)).mean()
    bad_pred = (df.prediction_raw == -1).mean()
    print(f"{name}invalid predictions: {bad_pred:.5f} | logprob sentinels: {bad_lp:.5f}")
    return bad_lp == 0 and bad_pred == 0

## Step 0: Integrity check

A failed label-token lookup in `get_confidence` defaults the logprob to -100, and
a failed decode sets `prediction_raw = -1`. Either would produce numbers that
look like findings. Check first.

In [3]:
gate = pd.read_parquet(RESULTS / "gate_corruption_merged.parquet")
assert integrity(gate)
print(f"{len(gate)} rows | "
      f"{gate[['dataset','mechanism','corruption','seed']].drop_duplicates().shape[0]} units | "
      f"mechanisms={sorted(gate.mechanism.unique())} | corruptions={sorted(gate.corruption.unique())}")

invalid predictions: 0.00000 | logprob sentinels: 0.00000
16000 rows | 64 units | mechanisms=['random', 'zero_shot'] | corruptions=[np.float64(0.0), np.float64(0.5), np.float64(1.0)]


## Step 1: The gate verdict

AUROC on the label-logprob margin is the primary statistic because it is
**threshold-free** — and the threshold turns out to be exactly what is broken
(Step 2). Paired Wilcoxon over seeds, comparing 0% against 100% corruption.

In [4]:
few = gate[gate.mechanism == "random"]
rows = []
for ds, g in few.groupby("dataset"):
    per = {c: gc.groupby("seed").apply(lambda x: roc_auc_score(x.label, margin(x)))
           for c, gc in g.groupby("corruption")}
    a0, a1 = per[0.0].to_numpy(), per[1.0].to_numpy()
    rows.append(dict(dataset=ds, auroc_c0=a0.mean(), auroc_c50=per[0.5].mean(),
                     auroc_c100=a1.mean(), delta=a0.mean() - a1.mean(),
                     seeds_same_dir=int((a0 > a1).sum()),
                     wilcoxon_p=stats.wilcoxon(a0, a1).pvalue))
pd.DataFrame(rows).sort_values("delta", ascending=False).round(3)

,dataset,auroc_c0,auroc_c50,auroc_c100,delta,seeds_same_dir,wilcoxon_p
2,anes,0.682,0.552,0.507,0.175,5,0.062
0,acsincome,0.682,0.692,0.670,0.012,3,0.625
1,acspubcov,0.517,0.515,0.518,-0.001,3,1.000
3,brfss_diabetes,0.592,0.631,0.617,-0.025,2,0.625


**ANES is the only dataset where flipping the labels destroys the signal**, and
it goes all the way to chance. The other three are flat to within ~0.025 with no
consistent direction across seeds.

Note `p = 0.062` with 5/5 seeds agreeing: that is the **floor** of a 5-seed
Wilcoxon test, not a near-miss. See Notebook 14 for the arithmetic. Five seeds
cannot support inferential claims; this is a descriptive result.

`acspubcov` sits at chance regardless of corruption, independently confirming
Notebook 10's no-headroom verdict with a real LLM.

## Step 2: The finding that outranks the gate

Raw accuracy is unusable on these class-imbalanced datasets (Notebook 10 §2), so
this uses balanced accuracy and the model's implied `P(positive)`.

In [5]:
diag = gate.groupby(["dataset", "mechanism"]).apply(lambda s: pd.Series({
    "posrate_raw": (s.prediction_raw == 1).mean(),
    "bacc_raw": balanced_accuracy_score(s.label, s.prediction_raw),
    "posrate_calibrated": (s.prediction == 1).mean(),
    "bacc_calibrated": balanced_accuracy_score(s.label, s.prediction),
    "auroc": roc_auc_score(s.label, margin(s)),
    "pyes_median": float(np.median(p_positive(s))),
    "pyes_iqr": float(np.diff(np.percentile(p_positive(s), [25, 75]))[0]),
})).round(3)
diag

posrate_raw  bacc_raw  posrate_calibrated  bacc_calibrated  auroc  pyes_median  pyes_iqr
dataset        mechanism                                                                                          
acsincome      random           0.996     0.501               0.435            0.680  0.671        0.867     0.087
               zero_shot        0.700     0.596               0.924            0.448  0.708        0.706     0.477
acspubcov      random           1.000     0.500               0.757            0.520  0.520        0.867     0.058
               zero_shot        0.992     0.510               1.000            0.500  0.489        0.755     0.092
anes           random           0.948     0.508               0.451            0.568  0.583        0.835     0.138
               zero_shot        0.496     0.688               0.988            0.499  0.716        0.500     0.406
brfss_diabetes random           1.000     0.500               0.789            0.573  0.565        0.924     0.035
               zero_shot        0.568     0.613               0.936            0.517  0.675        0.593     0.622

**With eight demonstrations the uncalibrated model answers *positive* to
95-100% of queries and balanced accuracy is exactly 0.500 — a constant
predictor.** Without demonstrations the same model is far better behaved on three
of four datasets (positive rate 0.50-0.70, balanced accuracy 0.60-0.69) — but
**not** on `acspubcov`, whose zero-shot arm is *already* saturated (0.992 / 0.510)
before a single demonstration is added. That dataset's constant-predictor
behaviour cannot be attributed to demonstrations at all; it is a second, distinct
pathology.

Two consequences:

1. **`REDESIGN_RATIONALE.md` §4.1 is falsified.** It attributed v1's 96-99%
   class-1 rate to the missing chat template. The template is now correctly
   applied (`prompt_version=v2_chat`) and the raw rate is *still* 95-100%. The
   bias is **demonstration-induced, not format-induced**.
2. **Composition is not the lever Notebook 10 assumed.** These demonstrations
   were `balanced` — four positive, four negative — and still produced a ~100%
   positive output rate. Demo label *counts* do not drive this model's output
   prior, which is exactly the assumption the `prior_only` surrogate was built
   on. That inference does not transfer.

## Step 3: Why contextual calibration rescues the few-shot arm but breaks zero-shot

Contextual calibration (Zhao et al. 2021) divides out what the model outputs for
a content-free prompt. That is valid only if the content-free response is
actually an estimate of the prompt-induced bias.

In [6]:
cal = gate.groupby(["dataset", "mechanism"]).apply(lambda s: pd.Series({
    "observed_margin": float(margin(s).mean()),
    "content_free_margin": float((s.logprob_1_cf - s.logprob_0_cf).mean()),
})).round(3)
cal

observed_margin  content_free_margin
dataset        mechanism                                      
acsincome      random               1.862                1.892
               zero_shot            0.354               -2.750
acspubcov      random               1.833                1.673
               zero_shot            1.155               -1.875
anes           random               1.329                1.719
               zero_shot            0.445               -2.250
brfss_diabetes random               2.471                2.258
               zero_shot            0.373               -3.125

For the **few-shot** arm the content-free margin closely matches the observed
margin, so dividing it out centres the decision and balanced accuracy goes from
0.500 to 0.56-0.66. That near-equality is the signature of a purely
prompt-induced bias.

For the **zero-shot** arm the content-free margin is strongly *negative*, because
its content-free prompt has no demonstrations *and* placeholder query values —
a near-empty prompt. The model's response to that is not a label-prior estimate,
and dividing by it makes zero-shot substantially worse.

**Do not apply contextual calibration to the zero-shot arm as implemented.** A
corrected content-free baseline must keep the demonstrations and blank only the
query. Notebook 13 finds the same pathology hitting the *few-shot* arm at 70B.

## Step 4: Gate S1's own criteria

Criterion (b) requires the calibrated zero-shot class-1 rate to land in
[0.35, 0.65]. Criterion (a) requires random-8 ID accuracy >= 0.60 — not testable
here, since this run used OOD queries only. Notebook 13 adds the ID split.

In [7]:
zs = gate[gate.mechanism == "zero_shot"].groupby("dataset").apply(lambda s: pd.Series({
    "posrate_raw": (s.prediction_raw == 1).mean(),
    "posrate_calibrated": (s.prediction == 1).mean(),
}))
zs["passes_S1b"] = zs.posrate_calibrated.between(0.35, 0.65)
zs.round(3)

,posrate_raw,posrate_calibrated,passes_S1b
dataset,,,
acsincome,0.700,0.924,False
acspubcov,0.992,1.000,False
anes,0.496,0.988,False
brfss_diabetes,0.568,0.936,False


Criterion (b) fails 4/4 — though partly via the Step 3 calibration artefact: on
**raw** rates `anes` and `brfss_diabetes` pass, `acsincome` marginally fails and
`acspubcov` fails badly. Gate S1's own model-escalation clause has therefore
fired, which is what motivates the 70B arm in Notebook 13.

## The confound this notebook cannot resolve

Corruption-insensitivity tracks **saturation** of the decision variable almost
perfectly: `anes` is the least saturated dataset (`pyes_iqr` largest) and the only
mover; `brfss_diabetes` is the most saturated and shows nothing.

So "the model ignores the labels" and "the decision variable has no room left to
register an effect" are *both* consistent with this data, and they imply opposite
next steps. Notebook 12 attacks the confound directly by trying to de-saturate
the output; Notebook 13 settles it with a model whose output is 6.5x less
saturated.